In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q3_data.csv')
df_c = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
print(f"Dataset shape: {df_c.shape}")
df_c.head()


In [ ]:
# Task 3: Write your code here:
df_c.info()

In [ ]:
# Task 4: Write your code here:
df_c.describe()

In [ ]:
# Task 1: Write your code here:

# Count how many missing values exist in each column
# isnull() creates a boolean mask
# sum() counts how many True values (i.e., missing)
missing_values = df_c.isnull().sum()
print("\nMissing values per column:")
print(missing_values)

# Percentage of missing values per column
missing_percentage = (missing_values / len(df_c)) * 100
print("\nMissing values percentage:")
print(missing_percentage)


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_c)



In [ ]:
# Task 3: Write your code here:
#no need for encoded

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_c.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_c[numerical_cols] = scaler.fit_transform(df_c[numerical_cols])
df_c.head()


In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df_c, "Target")

In [ ]:
# Task 1: Write your code here:
X = df_c.drop("Target", axis=1).astype(float)
y = df_c['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
%pip install catboost
from catboost import CatBoostClassifier
sklearn_models = {

  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}

all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}



In [ ]:
n_splits = 5 # K
from sklearn.model_selection import StratifiedKFold

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['precision'].append(precision)
    all_results[model_name]['recall'].append(recall)
    all_results[model_name]['f1'].append(f1)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  Accuracy:  {np.mean(all_results[model_name]['accuracy']):.4f}")
  print(f"  Precision: {np.mean(all_results[model_name]['precision']):.4f}")
  print(f"  Recall:    {np.mean(all_results[model_name]['recall']):.4f}")
  print(f"  F1-Score:  {np.mean(all_results[model_name]['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: